# NF_SIMURGH GMD — Inference & Calculator

Loads the best checkpoint from the **NF_SIMURGH** GMD FragmentDuplication run and
provides a ready-to-use ASE calculator.

> **This is a duplicate of `simurgh_gmd_inference.ipynb`, adapted for NF_SIMURGH.**
> The original is left untouched as a backup.

**Run:** `logs/nf_simurgh_gmd26/runs/2026-08-28_11-37-41_trajectory_nf_simurgh_fragment_duplication`
**Dataset:** GMD FragmentDuplication — O₂CₙH₂ₙ molecules, PBE/def2-SVP DFT

---

## Why this notebook needs its own calculator

NF_SIMURGH's unit of computation is the **fragment virtual node**: per-atom energies
from the ARACE rounds are pooled onto one virtual node per molecular fragment, and
those nodes then interact pairwise across the whole structure. Every graph therefore
has to carry a `fragment_index`.

`GOALCalculator` does not build one. Its `calculate()` calls
`AtomicGraph.from_ase(atoms, cutoff=..., dtype=..., head=...)`, and `from_ase`
defaults `compute_fragment_index=False` — so inference fails with:

> NF_SIMURGH computes on fragment virtual nodes, so every graph must carry a
> `'fragment_index'`. Set `data.compute_fragment_index: true` in the config ...

**That error is deliberate, not a bug.** A silent fallback to atom-level-only would
turn the model into `monolithic_arace` wearing NF_SIMURGH's name and report entirely
plausible numbers from the wrong model. Raising is the correct behaviour.

The fix is to attach the labels — using **exactly the settings the model was trained
with**. `NFSimurghCalculator` below reads them straight out of the checkpoint's own
config rather than hardcoding them, so the inference decomposition cannot silently
drift from the training one.

In [ ]:
from __future__ import annotations

from pathlib import Path

import torch

# Trigger GOAL registry population and Lightning checkpoint patch
import goal.ml.cli  # noqa: F401

from goal.ml.utils.calculator import GOALCalculator

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1 · Load the Best Checkpoint

The best checkpoint is selected by lowest validation loss.  
`GOALCalculator` reconstructs the backbone + head from the saved Hydra config
and loads the state dict automatically.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────
PROJECT_ROOT = Path("..")
RUN_DIR = PROJECT_ROOT / "logs" / "nf_simurgh_gmd26" / "runs" / "2026-08-28_11-37-41_trajectory_nf_simurgh_fragment_duplication"
CKPT_DIR = RUN_DIR / "checkpoints"

assert CKPT_DIR.is_dir(), f"Checkpoint directory not found: {CKPT_DIR.resolve()}"

# ── Device selection ──────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Checkpoint dir  : {CKPT_DIR.resolve()}")
print(f"Running on      : {DEVICE}")

## 1.1 · Fragment-aware calculator

`NFSimurghCalculator` subclasses `GOALCalculator` and overrides `calculate()` with one
change: the graph is built **with fragment labels attached**.

Two details matter more than they look:

1. **The fragment settings come from the checkpoint, not from this notebook.**
   `self._module.config` is the Hydra config the run was trained with. Reading
   `fragment_scheme`, `fragment_smarts`, `fragment_covalent_cutoff` and friends from
   there guarantees inference partitions molecules the same way training did.
   Hardcoding `scheme="connected"` here would give K=1 on every GMD acid — one virtual
   node, no pairs, the entire mechanism silently inert — and the model would still
   happily return numbers.

2. **`freeze_fragments`** (default `False`) pins the decomposition to the first
   structure seen. Off, fragments are recomputed per call, which is what training did.
   On, they are held fixed — which is what you want for MD and geometry optimisation,
   where a bond stretching past `fragment_covalent_cutoff` would otherwise re-perceive
   the topology mid-trajectory and put a step in the energy. See §4 and §5.

The override also surfaces `vnode_energies` / `vnode_interaction_energies` in
`calc.results`, so the fragment channel's contribution is inspectable.

> **Upstream note:** this whole class collapses to one line if `GOALCalculator` ever
> forwards the config's fragment settings into `from_ase()`. It is kept here in the
> notebook so nothing in `src/` had to change.

In [ ]:
from __future__ import annotations

import typing

from ase.calculators.calculator import all_changes
from ase.stress import full_3x3_to_voigt_6_stress

from goal.ml.data.graph import AtomicGraph


def _plain(value: typing.Any) -> typing.Any:
    """OmegaConf container → plain Python (from_ase does not accept DictConfig)."""
    if value is None:
        return None
    try:
        from omegaconf import OmegaConf

        if OmegaConf.is_config(value):
            return OmegaConf.to_object(value)
    except ImportError:
        pass
    return list(value) if isinstance(value, (list, tuple)) else value


class NFSimurghCalculator(GOALCalculator):
    """GOALCalculator that attaches the ``fragment_index`` NF_SIMURGH requires.

    Parameters
    ----------
    freeze_fragments : bool
        ``False`` (default) recomputes the decomposition on every call — the same
        behaviour as dataset construction during training.  ``True`` computes it once
        and reuses it, which is the right choice for MD / relaxation: it keeps the
        potential-energy surface continuous even if a bond wanders across
        ``fragment_covalent_cutoff``.
    """

    def __init__(self, *args: typing.Any, freeze_fragments: bool = False, **kwargs: typing.Any) -> None:
        super().__init__(*args, **kwargs)

        # Fragment settings straight from the checkpoint's own training config,
        # so inference cannot partition molecules differently from training.
        data_cfg = self._module.config.data
        if not bool(data_cfg.get("compute_fragment_index", False)):
            raise ValueError(
                "This checkpoint was trained with data.compute_fragment_index: false, "
                "so it cannot be an NF_SIMURGH run.  Use the plain GOALCalculator."
            )
        self.fragment_options: dict[str, typing.Any] = {
            "compute_fragment_index": True,
            "fragment_covalent_cutoff": float(data_cfg.get("fragment_covalent_cutoff", 1.8)),
            "fragment_scheme": str(data_cfg.get("fragment_scheme", "connected")),
            "fragment_charge": int(data_cfg.get("fragment_charge", 0)),
            "fragment_on_failure": str(data_cfg.get("fragment_on_failure", "fallback")),
            "fragment_smarts": _plain(data_cfg.get("fragment_smarts", None)),
            "fragment_keep_groups": _plain(data_cfg.get("fragment_keep_groups", None)),
        }

        self.freeze_fragments: bool = bool(freeze_fragments)
        self._frozen_fragment_index: torch.Tensor | None = None
        #: Labels used by the most recent ``calculate()`` — for inspection.
        self.last_fragment_index: torch.Tensor | None = None

    def reset_frozen_fragments(self) -> None:
        """Drop the pinned decomposition so the next call recomputes it."""
        self._frozen_fragment_index = None

    def calculate(
        self,
        atoms: typing.Any = None,
        properties: list[str] | None = None,
        system_changes: list[str] = all_changes,
    ) -> None:
        """Mirrors ``GOALCalculator.calculate`` — the ONLY change is that the
        graph is built with ``**self.fragment_options``."""
        if properties is None:
            properties = self.implemented_properties

        # Skip GOALCalculator.calculate, run ase.Calculator's bookkeeping directly.
        super(GOALCalculator, self).calculate(atoms, properties, system_changes)

        if self.atoms is None:
            raise RuntimeError("No atoms object set on calculator.")

        graph: AtomicGraph = AtomicGraph.from_ase(
            self.atoms,
            cutoff=self._cutoff,
            dtype=self.dtype,
            head=self.head,
            **self.fragment_options,          # ← the fix
        )

        if self.freeze_fragments:
            if (
                self._frozen_fragment_index is None
                or self._frozen_fragment_index.numel() != len(self.atoms)
            ):
                self._frozen_fragment_index = graph.fragment_index.clone()
            graph.fragment_index = self._frozen_fragment_index.clone()

        self.last_fragment_index = graph.fragment_index.detach().cpu().clone()
        graph = graph.to(self.device)

        with torch.enable_grad():
            predictions: dict[str, torch.Tensor] = self._module(graph)

        if "energy" in predictions:
            self.results["energy"] = predictions["energy"].detach().cpu().item()

        if "forces" in predictions:
            self.results["forces"] = predictions["forces"].detach().cpu().numpy()

        if "stress" in predictions:
            stress_np = predictions["stress"].detach().cpu().numpy()
            if stress_np.shape == (3, 3):
                self.results["stress"] = full_3x3_to_voigt_6_stress(stress_np)
            elif stress_np.shape == (6,):
                self.results["stress"] = stress_np
            else:
                self.results["stress"] = stress_np.flatten()[:6]

        # NF_SIMURGH-specific diagnostics — per-fragment totals and the learned
        # correction alone.  The correction is exactly zero at initialisation, so a
        # non-zero value here means the fragment channel is actually being used.
        for key in ("vnode_energies", "vnode_interaction_energies"):
            if key in predictions:
                self.results[key] = predictions[key].detach().cpu().numpy()


print("NFSimurghCalculator defined.")

In [ ]:
# ── Build the calculator ──────────────────────────────────────────
# NFSimurghCalculator, not GOALCalculator — see §1.1 for why.
# Passing the directory (FORMAT B) instead of a hardcoded filename means this
# cell always loads the true best checkpoint for this run, via
# checkpoint_state.json and the frozen source snapshot.

calc = NFSimurghCalculator(
    checkpoint_path=CKPT_DIR,
    checkpoint="best",
    device=DEVICE,
)

print(calc)
n_params = sum(p.numel() for p in calc._module.parameters())
print(f"Model parameters: {n_params:,}")
print()
print("Fragment settings recovered from the checkpoint:")
for key, value in calc.fragment_options.items():
    print(f"  {key:26s} = {value}")

# The 'Could not reconstruct loss components ...' warning above is expected and
# harmless: the loss is only needed for training, never for inference.

## 1.2 · Verify the mechanism is actually live

Two things worth confirming before trusting any number below.

**The decomposition matches training.** The config cuts `[CX4]-[CX4]` and
`[CX4]-[CX3]` bonds, which on these carboxylic acids should give one fragment per
CH₂/CH₃ group plus one for the COOH — K=4 for the C4 acid, K=10 for the C10 acid.
If you see K=1, the scheme fell back to `connected` and the virtual-node channel is
inert.

**The correction is non-zero.** `vnode_interaction_energies` is zero-initialised by
design, so a trained model showing values away from zero is the direct evidence that
the fragment channel learned to contribute rather than being ignored.

In [ ]:
import numpy as np
from ase.io import read as ase_read

_probe_dir = PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication" / "TestID"
_probe_files = sorted(_probe_dir.glob("*.traj"))

print(f"{'structure':>12s}  {'N':>3s}  {'K':>3s}  fragment sizes")
print("-" * 64)
for _path in _probe_files:
    _frame = ase_read(str(_path), index=0)
    _frame.calc = calc
    _frame.get_potential_energy()                      # triggers a forward pass
    _labels = calc.last_fragment_index
    _k = int(_labels.max()) + 1
    _sizes = torch.bincount(_labels).tolist()
    print(f"{_frame.get_chemical_formula():>12s}  {len(_frame):3d}  {_k:3d}  {_sizes}")

    if _k == 1:
        print("      ⚠  K=1 — no fragment pairs, the virtual-node channel is INERT here.")

# ── Is the learned correction actually doing anything? ─────────────────────
_vnode_int = calc.results.get("vnode_interaction_energies")
_vnode_tot = calc.results.get("vnode_energies")
print()
if _vnode_int is None:
    print("No vnode diagnostics returned — is this really an nf_simurgh checkpoint?")
else:
    print(f"Last structure — per-fragment energies (eV), K={len(_vnode_int)}:")
    print(f"  total       : {np.round(_vnode_tot, 4)}")
    print(f"  correction  : {np.round(_vnode_int, 4)}")
    print(f"  |correction| mean = {np.abs(_vnode_int).mean():.6f} eV")
    print(f"  correction as a fraction of |E_total| = "
          f"{np.abs(_vnode_int).sum() / abs(calc.results['energy']):.3e}")
    if np.abs(_vnode_int).max() < 1e-8:
        print("  ⚠  correction is still ~zero — the fragment channel never activated "
              "during training (it is zero-initialised by design).")
    else:
        print("  ✓ non-zero — the fragment channel is contributing.")

## 1.5 · Quick test

In [ ]:
import ase.io
TRAJ_PATH  = "/home/masoud/Projects/SCAI/GMD/data/CarboxylicAcids/O2C7H14_PBE.traj"
# Read all frames and pick a middle snapshot
traj  = ase.io.read(TRAJ_PATH, index=":")
frame = traj[1800]

# Reference (DFT) forces are stored in the SinglePointCalculator attached to each frame
ref_forces = frame.get_forces().copy()

print(f"Trajectory length : {len(traj)} frames")
print(f"Using frame index : {len(traj) // 2}")
print(f"Atoms             : {frame.get_chemical_formula()}  ({len(frame)} atoms)")
print(f"Ref forces shape  : {ref_forces.shape}")

atoms_pred = frame.copy()
atoms_pred.calc = calc
pred_forces = atoms_pred.get_forces()

print(f"Predicted forces shape: {pred_forces.shape}")

In [ ]:
import numpy as np
# Per-atom error: L2 norm of (pred - ref) force vector
atom_errors = np.linalg.norm(pred_forces - ref_forces, axis=1)
component_mae = np.abs(pred_forces - ref_forces).mean()  
# newton_error = pred_forces.norm(dim=-1).mean().item() - ref_forces.norm(dim=-1).mean().item()

print(f"Force errors (eV/Å):")
print(f"  min  = {atom_errors.min():.4f}")
print(f"  max  = {atom_errors.max():.4f}")
print(f"  mean = {atom_errors.mean():.4f}")
print(f"  component MAE = {component_mae:.4f}")
# print(f"  newton = {newton_error:.4f}")

# Normalize to [0, 1]  (0 = best / white, 1 = worst / red)
err_range   = atom_errors.max() - atom_errors.min()
norm_errors = (atom_errors - atom_errors.min()) / (err_range + 1e-10)

print(f"Total force error (eV/Å): {atom_errors.sum():.4f}")

In [ ]:
import torch
from goal.ml.training.metrics import mlip_metrics

# Convert numpy arrays to tensors; single structure → all atoms belong to graph 0
pred_f_t  = torch.as_tensor(pred_forces, dtype=torch.float32)
ref_f_t   = torch.as_tensor(ref_forces,  dtype=torch.float32)
batch_idx = torch.zeros(len(pred_forces), dtype=torch.long)

# mlip_metrics expects a predictions dict and a batch-like object with
# .forces and .batch — wrap the single-structure arrays minimally.
class _SingleStructureBatch:
    forces = ref_f_t
    batch  = batch_idx

metrics = mlip_metrics({"forces": pred_f_t}, _SingleStructureBatch())
newton_violation = metrics["newton_violation"].item()

net_force = pred_forces.sum(axis=0)  # same as scatter_sum over a single graph
magnitude_bias = pred_f_t.norm(dim=-1).mean().item() - ref_f_t.norm(dim=-1).mean().item()

print(f"Newton violation (project formula) : {newton_violation:.4e} eV/Å")
print(f"  net force vector : [{net_force[0]:+.4e}  {net_force[1]:+.4e}  {net_force[2]:+.4e}] eV/Å")
print()
print(f"Magnitude bias (old commented formula) : {magnitude_bias:+.4e} eV/Å")
print(f"  (mean|F_pred| − mean|F_ref| — NOT a Newton metric)")

## 2 · Quick Sanity Check

Attach the calculator to an ASE `Atoms` object and verify energy + forces.

In [ ]:
import numpy as np
from ase import Atoms
from ase import io

# Water molecule — H, O, H all within the training element set {H, C, O}
atoms: Atoms = io.read("../data/GMD/FragmentDuplication/TestOOD/O4C4H6_PBE.traj", index=0)
atoms.calc = calc

energy: float = atoms.get_potential_energy()   # eV
forces: np.ndarray = atoms.get_forces()         # eV/Å,  shape (N, 3)

print(f"Formula  : {atoms.get_chemical_formula()}")
print(f"Energy   : {energy:.4f} eV")
print()
print("Forces (eV/Å):")
for sym, f in zip(atoms.get_chemical_symbols(), forces):
    print(f"  {sym:2s}  [{f[0]:+8.4f}  {f[1]:+8.4f}  {f[2]:+8.4f}]")

## 3 · Evaluate on a Test-Set Structure

Load one trajectory from the OOD test split and compare predicted vs. DFT values.

In [ ]:
from ase.io import read as ase_read

TEST_DIR = PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication" / "TestOOD"
traj_files = sorted(TEST_DIR.glob("*.traj"))
print(f"Found {len(traj_files)} test trajectory files in {TEST_DIR.name}/")

# Pick the first structure of the first file
sample_frame: Atoms = ase_read(str(traj_files[0]), index=0)
print(f"\nSample structure: {sample_frame.get_chemical_formula()} ({len(sample_frame)} atoms)")

# DFT reference values
dft_energy: float = sample_frame.get_potential_energy()   # already stored in .traj
dft_forces: np.ndarray = sample_frame.get_forces()

# SIMURGH prediction
sample_frame.calc = calc
pred_energy: float = sample_frame.get_potential_energy()
pred_forces: np.ndarray = sample_frame.get_forces()

n_atoms = len(sample_frame)
energy_mae_per_atom = abs(pred_energy - dft_energy) / n_atoms
forces_mae = np.abs(pred_forces - dft_forces).mean()

print(f"\n{'':20s} {'DFT':>12s} {'SIMURGH':>12s}")
print(f"{'Energy (eV)':20s} {dft_energy:12.4f} {pred_energy:12.4f}")
print(f"{'Energy MAE/atom':20s} {'':12s} {energy_mae_per_atom:12.4f} eV/atom")
print(f"{'Forces MAE (eV/Å)':20s} {'':12s} {forces_mae:12.4f}")

In [ ]:
from ase.io import read as ase_read

TEST_DIR = PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication" / "TestID"
traj_files = sorted(TEST_DIR.glob("*.traj"))
print(f"Found {len(traj_files)} test trajectory files in {TEST_DIR.name}/")

# Pick the first structure of the first file
sample_frame: Atoms = ase_read(str(traj_files[0]), index=65)
print(f"\nSample structure: {sample_frame.get_chemical_formula()} ({len(sample_frame)} atoms)")

# DFT reference values
dft_energy: float = sample_frame.get_potential_energy()   # already stored in .traj
dft_forces: np.ndarray = sample_frame.get_forces()

# SIMURGH prediction
sample_frame.calc = calc
pred_energy: float = sample_frame.get_potential_energy()
pred_forces: np.ndarray = sample_frame.get_forces()

n_atoms = len(sample_frame)
energy_mae_per_atom = abs(pred_energy - dft_energy) / n_atoms
forces_mae = np.abs(pred_forces - dft_forces).mean()

print(f"\n{'':20s} {'DFT':>12s} {'SIMURGH':>12s}")
print(f"{'Energy (eV)':20s} {dft_energy:12.4f} {pred_energy:12.4f}")
print(f"{'Energy MAE/atom':20s} {'':12s} {energy_mae_per_atom:12.4f} eV/atom")
print(f"{'Forces MAE (eV/Å)':20s} {'':12s} {forces_mae:12.4f}")

In [ ]:
# Load MACE and predict on a copy so the original SinglePointCalculator is preserved
sample_frame: Atoms = ase_read(str(traj_files[0]), index=65)
ref_forces = sample_frame.get_forces().copy()
atoms_pred = sample_frame.copy()
atoms_pred.calc = calc
pred_forces = atoms_pred.get_forces()

print(f"Predicted forces shape: {pred_forces.shape}")

In [ ]:
# Per-atom error: L2 norm of (pred - ref) force vector
atom_errors = np.linalg.norm(pred_forces - ref_forces, axis=1)

print(f"Force errors (eV/Å):")
print(f"  min  = {atom_errors.min():.4f}")
print(f"  max  = {atom_errors.max():.4f}")
print(f"  mean = {atom_errors.mean():.4f}")

# Normalize to [0, 1]  (0 = best / white, 1 = worst / red)
err_range   = atom_errors.max() - atom_errors.min()
norm_errors = (atom_errors - atom_errors.min()) / (err_range + 1e-10)

print(f"Total force error (eV/Å): {atom_errors.sum():.4f}")

sum_pred_forces = pred_forces.sum(axis=0)
sum_ref_forces  = ref_forces.sum(axis=0)
sum_diff        = sum_pred_forces - sum_ref_forces

print("Summed forces (vector):")
print(f"  predicted : {sum_pred_forces}")
print(f"  reference : {sum_ref_forces}")
print(f"  difference: {sum_diff}")

print("\nSummed force magnitudes:")
print(f"  |sum_pred| = {np.linalg.norm(sum_pred_forces):.16f}")
print(f"  |sum_ref|  = {np.linalg.norm(sum_ref_forces):.16f}")
print(f"  |sum_diff| = {np.linalg.norm(sum_diff):.16f}")

In [ ]:
import torch.nn.functional as F_func
import py3Dmol

def error_to_hex(t: float) -> str:
    """Map normalised error t∈[0,1] → hex colour: white (t=0) to red (t=1)."""
    r = 255
    g = int(255 * (1.0 - t))
    b = int(255 * (1.0 - t))
    return f"#{r:02x}{g:02x}{b:02x}"

atom_colors = [error_to_hex(t) for t in norm_errors]

# Quick sanity check — print first few atoms with their error and color
for i, (sym, err, col) in enumerate(zip(sample_frame.get_chemical_symbols(), atom_errors, atom_colors)):
    print(f"  atom {i:2d}  {sym}  error={err:.4f}  color={col}")

# ── Two-sphere helpers ─────────────────────────────────────────────────────────
# inner sphere : relative magnitude error  ||F_pred|-|F_ref|| / ||F_ref||   (white → pink)
# outer shell  : cosine similarity         cos(F_pred, F_ref)
#                cos=-1 → YELLOW  (opacity ~0.85)
#                cos= 0 → RED     (opacity ~0.55)
#                cos=+1 → WHITE   (opacity ~0.25 — faint translucent halo)
#                Always rendered so the shell is visible at every atom.
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

# white → pink — relative magnitude error
cmap_mag = LinearSegmentedColormap.from_list("mag_err", ["#ffffff", "#ffb6d5", "#ff1493"])

# Parameterise by t = (1 - cos) / 2  ∈ [0, 1]   (t=0 at cos=+1, t=1 at cos=-1)
#   t=0   (cos=+1): white
#   t=0.5 (cos= 0): red
#   t=1   (cos=-1): yellow
OUTER_WHITE  = "#ffffff"
OUTER_RED    = "#e60026"
OUTER_YELLOW = "#ffd000"
cmap_dir = LinearSegmentedColormap.from_list(
    "dir_err",
    [(0.0, OUTER_WHITE), (0.5, OUTER_RED), (1.0, OUTER_YELLOW)],
)
OUTER_OPACITY_MIN = 0.25
OUTER_OPACITY_MAX = 0.85

def two_sphere_colors(pred_forces, ref_forces, mag_vmax=None):
    """Return (inner_colors, outer_colors, outer_opacities, rel_mag, cos_sim)."""
    vp = torch.as_tensor(pred_forces, dtype=torch.float32)
    vr = torch.as_tensor(ref_forces,  dtype=torch.float32)
    cos_sim = F_func.cosine_similarity(vp, vr, dim=1).cpu().numpy()
    cos_sim = np.nan_to_num(cos_sim, nan=0.0)

    n_pred = np.linalg.norm(pred_forces, axis=1)
    n_ref  = np.linalg.norm(ref_forces,  axis=1)
    rel_mag = np.abs(n_pred - n_ref) / (n_ref + 1e-10)

    vmax = mag_vmax if mag_vmax is not None else max(float(rel_mag.max()), 1e-10)
    inner_colors    = [mcolors.to_hex(cmap_mag(min(float(m) / vmax, 1.0))) for m in rel_mag]

    t_dir           = [(1.0 - float(c)) / 2.0 for c in cos_sim]                 # 0..1
    outer_colors    = [mcolors.to_hex(cmap_dir(t)) for t in t_dir]
    span            = OUTER_OPACITY_MAX - OUTER_OPACITY_MIN
    outer_opacities = [OUTER_OPACITY_MIN + t * span for t in t_dir]
    return inner_colors, outer_colors, outer_opacities, rel_mag, cos_sim

def add_two_spheres(view, positions, inner_colors, outer_colors, outer_opacities,
                    r_inner=0.22, r_outer=0.42):
    """Two concentric spheres per atom; outer shell ALWAYS rendered."""
    for i, pos in enumerate(positions):
        x, y, z = float(pos[0]), float(pos[1]), float(pos[2])
        view.addSphere({
            "center":  {"x": x, "y": y, "z": z},
            "radius":  r_outer,
            "color":   outer_colors[i],
            "opacity": float(outer_opacities[i]),
        })
        view.addSphere({
            "center": {"x": x, "y": y, "z": z},
            "radius": r_inner,
            "color":  inner_colors[i],
        })


In [ ]:
# Serialise the frame to XYZ string for py3Dmol
import io as sio
buf = sio.StringIO()
io.write(buf, sample_frame, format='xyz')
xyz_str = buf.getvalue()

In [ ]:
from ase.neighborlist import natural_cutoffs, NeighborList

symbols   = sample_frame.get_chemical_symbols()
positions = sample_frame.get_positions()

view = py3Dmol.view(width=800, height=550)
view.addModel(xyz_str, 'xyz')

# Suppress py3Dmol's own bond drawing — we draw bonds manually below
view.setStyle({}, {})

# Compute bonds with ASE natural cutoffs (covalent radii, 10% tolerance)
cutoffs = natural_cutoffs(sample_frame, mult=1.1)
nl = NeighborList(cutoffs, self_interaction=False, bothways=False)
nl.update(sample_frame)

for i in range(len(sample_frame)):
    indices, _ = nl.get_neighbors(i)
    for j in indices:
        if j > i:
            p1, p2 = positions[i], positions[j]
            view.addCylinder({
                'start':   {'x': float(p1[0]), 'y': float(p1[1]), 'z': float(p1[2])},
                'end':     {'x': float(p2[0]), 'y': float(p2[1]), 'z': float(p2[2])},
                'radius':  0.08,
                'color':   '#888888',
                'fromCap': True,
                'toCap':   True,
            })

# Two-sphere atoms: inner = magnitude error (white→pink), outer shell = cos sim (yellow/red, opacity ∝ |cos|)
inner_full, outer_full, opac_full, rel_mag_full, cos_full = two_sphere_colors(pred_forces, ref_forces)
add_two_spheres(view, positions, inner_full, outer_full, opac_full)

for sym, pos in zip(symbols, positions):
    view.addLabel(sym, {
        'position':       {'x': float(pos[0]), 'y': float(pos[1]), 'z': float(pos[2])},
        'fontSize':       12,
        'fontColor':      'black',
        'showBackground': False,
        'alignment':      'center',
    })

view.zoomTo()
view.show()


In [ ]:
# Two colourbars: magnitude error (inner sphere) and cosine similarity (outer shell)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable

# Inner: relative magnitude error
fig, ax = plt.subplots(figsize=(5, 0.6))
fig.subplots_adjust(bottom=0.5)
norm_mag = mcolors.Normalize(vmin=0.0, vmax=float(rel_mag_full.max()))
cb_mag   = fig.colorbar(ScalarMappable(norm=norm_mag, cmap=cmap_mag), cax=ax,
                        orientation='horizontal')
cb_mag.set_label('Inner sphere — relative magnitude error '
                 r'$|\,\|F_{pred}\|-\|F_{ref}\|\,| / \|F_{ref}\|$', fontsize=10)
plt.show()

# Outer: cosine similarity ∈ [-1, +1]
#   cos=-1 → yellow (opaque)
#   cos= 0 → red    (semi-translucent)
#   cos=+1 → white  (faint translucent halo)
cmap_dir_cb = mcolors.LinearSegmentedColormap.from_list(
    'dir_err_cb',
    [(0.0, OUTER_YELLOW), (0.5, OUTER_RED), (1.0, OUTER_WHITE)],   # cos=-1 → 0 → +1
)
fig, ax = plt.subplots(figsize=(5, 0.6))
fig.subplots_adjust(bottom=0.5)
norm_dir = mcolors.Normalize(vmin=-1.0, vmax=1.0)
cb_dir   = fig.colorbar(ScalarMappable(norm=norm_dir, cmap=cmap_dir_cb), cax=ax,
                        orientation='horizontal')
cb_dir.set_label(r'Outer shell — cosine similarity  $\cos(F_{pred}, F_{ref})$'
                 r'   (opacity grows from 0.25 at $\cos=+1$ to 0.85 at $\cos=-1$)',
                 fontsize=10)
plt.show()


In [ ]:
# Full evaluation over EVERY frame of both test splits.
#
# One pass, rich per-structure records — the report cells below all consume
# `all_results` and never re-run the model.  The previous version printed one line
# per structure (2800 lines) and a single mean; that hid a 20x spread across
# molecules, which is the whole story on the OOD split.

split_dirs = {
    "OOD": PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication" / "TestOOD",
    "ID":  PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication" / "TestID",
}


def evaluate_split(split_dir):
    """Every frame of every trajectory in *split_dir*, one record each."""
    records = []
    for traj_path in sorted(split_dir.glob("*.traj")):
        for idx, frame in enumerate(ase_read(str(traj_path), index=":")):
            dft_energy = frame.get_potential_energy()
            dft_forces = frame.get_forces().copy()
            n_atoms = len(frame)

            frame.calc = calc
            pred_energy = frame.get_potential_energy()
            pred_forces = frame.get_forces()

            delta = pred_forces - dft_forces
            err_norm = np.linalg.norm(delta, axis=1)            # (N,) per-atom
            mag_pred = np.linalg.norm(pred_forces, axis=1)
            mag_ref = np.linalg.norm(dft_forces, axis=1)
            cos_sim = np.einsum("ij,ij->i", pred_forces, dft_forces) / (
                mag_pred * mag_ref + 1e-12
            )

            n_oxygen = sum(1 for z in frame.get_atomic_numbers() if z == 8)
            n_frags = int(calc.last_fragment_index.max()) + 1

            records.append({
                "file": traj_path.name,
                "formula": frame.get_chemical_formula(),
                "index": idx,
                "n_atoms": n_atoms,
                "n_fragments": n_frags,
                "n_carboxyl": n_oxygen // 2,            # O2 -> 1 COOH, O4 -> 2
                "energy_dft": dft_energy,
                "energy_pred": pred_energy,
                "energy_err": pred_energy - dft_energy,             # signed
                "energy_err_per_atom": (pred_energy - dft_energy) / n_atoms,
                "forces_mae": float(np.abs(delta).mean()),
                "forces_rmse": float(np.sqrt((delta ** 2).mean())),
                "forces_err_max": float(err_norm.max()),
                "forces_cos_mean": float(cos_sim.mean()),
                "forces_cos_min": float(cos_sim.min()),
                "forces_mag_pred_mean": float(mag_pred.mean()),
                "forces_mag_ref_mean": float(mag_ref.mean()),
                "net_force": float(np.linalg.norm(pred_forces.sum(axis=0))),
                "vnode_int_abs_mean": float(
                    np.abs(calc.results["vnode_interaction_energies"]).mean()
                ),
            })
    return records


all_results = {}
for split_name, split_dir in split_dirs.items():
    print(f"evaluating {split_name} ...", flush=True)
    all_results[split_name] = evaluate_split(split_dir)
    print(f"  {len(all_results[split_name]):,} structures", flush=True)


def col(records, key):
    return np.array([r[key] for r in records])


print()
for split_name, records in all_results.items():
    f_mae = col(records, "forces_mae")
    e_pa = np.abs(col(records, "energy_err_per_atom"))
    print(f"=== {split_name} — {len(records):,} structures, "
          f"{col(records, 'n_atoms').sum():,} atoms ===")
    print(f"  Energy MAE / atom : {e_pa.mean():.6f} eV/atom")
    print(f"  Forces MAE        : {f_mae.mean():.6f} eV/Å")
    print(f"  Force cosine      : {col(records, 'forces_cos_mean').mean():.6f}")
    print(f"  |F_pred|/|F_ref|  : "
          f"{col(records, 'forces_mag_pred_mean').mean() / col(records, 'forces_mag_ref_mean').mean():.4f}")
    print(f"  Newton |ΣF| max   : {col(records, 'net_force').max():.3e} eV/Å")
    print()

## 6.5 · Full test-set report — every structure, both splits

`all_results` now holds one record per structure for **all 1,400 OOD and all 1,400 ID
frames**. The cells below slice it; none of them re-runs the model.

The single aggregate number is not the story here. On OOD it hides a **20× spread
across molecules**, so the per-molecule table is the one to read first.

In [ ]:
# ── Per-molecule breakdown ───────────────────────────────────────────
for split_name in ("ID", "OOD"):
    records = all_results[split_name]
    by_file: dict[str, list] = {}
    for r in records:
        by_file.setdefault(r["file"], []).append(r)

    print(f"=== {split_name} — per molecule ===")
    print(f"{'formula':>10s} {'n':>5s} {'N':>4s} {'K':>3s} {'COOH':>5s} "
          f"{'F_MAE':>9s} {'F_med':>9s} {'F_p90':>9s} {'|E|/atom':>10s} "
          f"{'cos':>9s} {'|Fp|/|Fr|':>10s}")
    print("-" * 96)
    for name in sorted(by_file, key=lambda f: by_file[f][0]["n_atoms"]):
        rs = by_file[name]
        f_mae = col(rs, "forces_mae")
        print(f"{rs[0]['formula']:>10s} {len(rs):5d} {rs[0]['n_atoms']:4d} "
              f"{rs[0]['n_fragments']:3d} {rs[0]['n_carboxyl']:5d} "
              f"{f_mae.mean():9.4f} {np.median(f_mae):9.4f} "
              f"{np.percentile(f_mae, 90):9.4f} "
              f"{np.abs(col(rs, 'energy_err_per_atom')).mean():10.4f} "
              f"{col(rs, 'forces_cos_mean').mean():9.5f} "
              f"{col(rs, 'forces_mag_pred_mean').mean() / col(rs, 'forces_mag_ref_mean').mean():10.4f}")
    print()

In [ ]:
# ── Error distribution, not just the mean ────────────────────────────
print(f"{'':22s} {'ID':>14s} {'OOD':>14s}")
print("-" * 52)
_id, _ood = all_results["ID"], all_results["OOD"]
for label, key, fn in [
    ("forces MAE  p10",   "forces_mae",          lambda a: np.percentile(a, 10)),
    ("forces MAE  median","forces_mae",          lambda a: np.median(a)),
    ("forces MAE  mean",  "forces_mae",          lambda a: a.mean()),
    ("forces MAE  p90",   "forces_mae",          lambda a: np.percentile(a, 90)),
    ("forces MAE  p99",   "forces_mae",          lambda a: np.percentile(a, 99)),
    ("forces MAE  max",   "forces_mae",          lambda a: a.max()),
    ("cos  mean",         "forces_cos_mean",     lambda a: a.mean()),
    ("cos  worst atom",   "forces_cos_min",      lambda a: a.min()),
    ("signed E/atom mean","energy_err_per_atom", lambda a: a.mean()),
    ("|E|/atom median",   "energy_err_per_atom", lambda a: np.median(np.abs(a))),
]:
    print(f"{label:22s} {fn(col(_id, key)):14.6f} {fn(col(_ood, key)):14.6f}")

# How much of OOD is actually fine?
ood_f = col(_ood, "forces_mae")
id_p99 = np.percentile(col(_id, "forces_mae"), 99)
good = ood_f <= 10 * id_p99
print(f"\nOOD frames within 10x the ID p99 ({10 * id_p99:.4f} eV/Å): "
      f"{good.sum():,} / {len(ood_f):,}  ({100 * good.mean():.1f}%)")
print("→ the OOD failure is concentrated, not uniform: a large minority of frames "
      "are predicted about as well as in-distribution ones.")

# ── Worst structures ─────────────────────────────────────────────────
print("\n=== 10 worst OOD structures by force MAE ===")
print(f"{'formula':>10s} {'frame':>6s} {'F_MAE':>9s} {'F_max':>9s} {'cos':>9s} {'E/atom':>10s}")
for r in sorted(_ood, key=lambda r: -r["forces_mae"])[:10]:
    print(f"{r['formula']:>10s} {r['index']:6d} {r['forces_mae']:9.4f} "
          f"{r['forces_err_max']:9.4f} {r['forces_cos_mean']:9.5f} "
          f"{r['energy_err_per_atom']:10.4f}")

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

# §5's plot cell calls matplotlib.use("Agg"); if it has already run, figures here
# would render to a file instead of inline.  Restore the notebook backend.
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# (a) per-molecule force MAE
ax = axes[0, 0]
for split_name, colour in (("ID", "#2196F3"), ("OOD", "#E53935")):
    by_file = {}
    for r in all_results[split_name]:
        by_file.setdefault(r["file"], []).append(r)
    order = sorted(by_file, key=lambda f: by_file[f][0]["n_atoms"])
    xs = [by_file[f][0]["n_atoms"] for f in order]
    ys = [col(by_file[f], "forces_mae").mean() for f in order]
    ax.plot(xs, ys, "o-", color=colour, label=split_name)
ax.set_yscale("log"); ax.set_xlabel("atoms per molecule")
ax.set_ylabel("force MAE (eV/Å)"); ax.set_title("(a) Force MAE by molecule size")
ax.grid(alpha=.3); ax.legend()

# (b) distribution of per-structure force MAE
ax = axes[0, 1]
bins = np.logspace(-3, 0.3, 60)
ax.hist(col(all_results["ID"], "forces_mae"), bins=bins, alpha=.65,
        color="#2196F3", label="ID")
ax.hist(col(all_results["OOD"], "forces_mae"), bins=bins, alpha=.65,
        color="#E53935", label="OOD")
ax.set_xscale("log"); ax.set_xlabel("per-structure force MAE (eV/Å)")
ax.set_ylabel("structures"); ax.set_title("(b) Error distribution")
ax.grid(alpha=.3); ax.legend()

# (c) force-magnitude parity — the systematic OOD bias
ax = axes[1, 0]
for split_name, colour in (("ID", "#2196F3"), ("OOD", "#E53935")):
    ax.scatter(col(all_results[split_name], "forces_mag_ref_mean"),
               col(all_results[split_name], "forces_mag_pred_mean"),
               s=5, alpha=.3, color=colour, label=split_name)
lim = [0, max(col(all_results["OOD"], "forces_mag_pred_mean").max(), 3.0)]
ax.plot(lim, lim, "k--", lw=1, label="parity")
ax.set_xlabel("mean |F_ref| (eV/Å)"); ax.set_ylabel("mean |F_pred| (eV/Å)")
ax.set_title("(c) Force magnitude — OOD sits above parity"); ax.grid(alpha=.3); ax.legend()

# (d) cosine similarity
ax = axes[1, 1]
ax.hist(col(all_results["ID"], "forces_cos_mean"), bins=np.linspace(0.3, 1.0, 60),
        alpha=.65, color="#2196F3", label="ID")
ax.hist(col(all_results["OOD"], "forces_cos_mean"), bins=np.linspace(0.3, 1.0, 60),
        alpha=.65, color="#E53935", label="OOD")
ax.set_xlabel("mean cos(F_pred, F_ref)"); ax.set_ylabel("structures")
ax.set_title("(d) Force direction"); ax.grid(alpha=.3); ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# ── Persist the full per-structure table ─────────────────────────────
import csv

out_path = RUN_DIR / "test_report.csv"
fieldnames = list(all_results["ID"][0].keys())
with open(out_path, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=["split"] + fieldnames)
    writer.writeheader()
    for split_name, records in all_results.items():
        for r in records:
            writer.writerow({"split": split_name, **r})

total = sum(len(r) for r in all_results.values())
print(f"Wrote {total:,} per-structure rows -> {out_path.resolve()}")
print(f"Columns: {', '.join(['split'] + fieldnames)}")

## 4 · Geometry Optimisation

Relax a perturbed structure to the nearest energy minimum with ASE's BFGS.

**Fragments are pinned for this section.** With `freeze_fragments=False` the
decomposition is re-perceived from the current geometry at every step; if a bond
crosses `fragment_covalent_cutoff` (1.8 Å) the molecule can repartition mid-run, which
puts a step in `E` and makes `F = -∂E/∂r` inconsistent with it right where the
optimiser is trying to converge. Pinning keeps the surface smooth. It is also the more
defensible physics: fragments are a *chemical* partitioning, and relaxing a molecule
is not supposed to change which group each atom belongs to.

In [ ]:
from ase.optimize import BFGS

# Pin the fragment decomposition for the whole relaxation — see the note above.
calc.freeze_fragments = True
calc.reset_frozen_fragments()

# Fresh copy — perturb positions slightly to start away from equilibrium
opt_atoms: Atoms = sample_frame.copy()
rng = np.random.default_rng(seed=42)
opt_atoms.positions += rng.normal(scale=0.05, size=opt_atoms.positions.shape)
opt_atoms.calc = calc

e0 = opt_atoms.get_potential_energy()
f0_max = np.abs(opt_atoms.get_forces()).max()
k_start = int(calc.last_fragment_index.max()) + 1
print(f"Fragments pinned : K={k_start}  sizes={torch.bincount(calc.last_fragment_index).tolist()}")
print(f"Initial energy   : {e0:.4f} eV")
print(f"Initial max force: {f0_max:.4f} eV/Å")

opt = BFGS(opt_atoms, logfile=None)
converged = opt.run(fmax=0.05, steps=100)

ef = opt_atoms.get_potential_energy()
ff_max = np.abs(opt_atoms.get_forces()).max()
print(f"\nFinal energy     : {ef:.4f} eV  (ΔE = {ef - e0:.4f} eV)")
print(f"Final max force  : {ff_max:.4f} eV/Å")
print(f"Steps taken      : {opt.nsteps}")
print(f"Converged        : {converged}")

calc.freeze_fragments = False
calc.reset_frozen_fragments()

## 5 · Short MD Simulation

NVT Langevin dynamics at 300 K on the NF_SIMURGH potential.

Fragments are pinned here for the same reason as §4, and it matters more in MD: at
300 K bond lengths fluctuate continuously, so a C–C bond sampling past the 1.8 Å
covalent cutoff would repartition the molecule mid-trajectory. That is a discontinuity
in the potential-energy surface, which shows up as spurious heating rather than as an
obvious error.

The cell below also *checks* the assumption — it recomputes the free-running
decomposition at the final geometry and reports whether it drifted.

In [ ]:
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

md_atoms: Atoms = sample_frame.copy()

# Pin fragments for the trajectory — see the note above.
calc.freeze_fragments = True
calc.reset_frozen_fragments()
md_atoms.calc = calc

MaxwellBoltzmannDistribution(md_atoms, temperature_K=300)

dyn = Langevin(
    md_atoms,
    timestep=0.5 * units.fs,
    temperature_K=300,
    friction=0.01 / units.fs,
)

md_energies: list[float] = []
dyn.attach(lambda: md_energies.append(md_atoms.get_potential_energy()), interval=1)

print("Running 50-step Langevin MD at 300 K ...")
dyn.run(steps=50)
print(f"Done — {len(md_energies)} energy snapshots collected")
print(f"Energy range: {min(md_energies):.4f} → {max(md_energies):.4f} eV")

# ── Did pinning actually matter? Recompute freely at the final geometry. ──
frozen_labels = calc.last_fragment_index.clone()
calc.freeze_fragments = False
calc.reset_frozen_fragments()
md_atoms.calc = calc
md_atoms.get_potential_energy()
free_labels = calc.last_fragment_index

if torch.equal(frozen_labels, free_labels):
    print("\nFragment check : topology stable across the trajectory "
          "(pinned == freely recomputed).")
else:
    print(f"\nFragment check : ⚠ decomposition DRIFTED — "
          f"K {int(frozen_labels.max()) + 1} → {int(free_labels.max()) + 1}. "
          f"Pinning was load-bearing here; without it this trajectory would have "
          f"crossed an energy discontinuity.")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(md_energies, linewidth=1.5, color="#2196F3")
ax.set_xlabel("MD step")
ax.set_ylabel("Potential energy (eV)")
ax.set_title(f"SIMURGH MD — {md_atoms.get_chemical_formula()} @ 300 K")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 6 · Batch Inference on the Test Set

Run the model directly on PyG batches for fast bulk evaluation (no ASE overhead).

In [ ]:
import os

import torch
from goal.ml.data.datamodule import GOALDataModule
from goal.ml.training.archive import resolve_checkpoint

# The saved config stores paths as `${oc.env:PROJECT_ROOT,${oc.env:PWD}}/data/...`
# and OmegaConf resolves that lazily, i.e. NOW.  Running from notebooks/ makes PWD
# the notebook directory, so the data dirs would resolve to notebooks/data/GMD/...
# and the datamodule raises FileNotFoundError.  Pin PROJECT_ROOT to the repo root
# before the config is read.
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT.resolve())
print(f"PROJECT_ROOT    : {os.environ['PROJECT_ROOT']}")

# Resolve the same best checkpoint the calculator uses
best_ckpt_path = resolve_checkpoint(CKPT_DIR, "best")
print(f"Best checkpoint : {best_ckpt_path.name}")

raw_ckpt = torch.load(str(best_ckpt_path), map_location="cpu", weights_only=False)
cfg = raw_ckpt["hyper_parameters"]["config"]

# The datamodule reads data.compute_fragment_index from this same config, so the
# batch path attaches fragment labels on its own — no calculator involved, and no
# extra work needed here.
print(f"compute_fragment_index : {cfg.data.compute_fragment_index}")
print(f"fragment_scheme        : {cfg.data.fragment_scheme}")

dm = GOALDataModule(cfg)
dm.prepare_data()
dm.setup("test")
test_loader = dm.test_dataloader()

print(f"Test set size : {len(dm.data_test):,} structures")
print(f"Batch size    : {cfg.data.batch_size}")
print(f"Num batches   : {len(test_loader)}")

In [ ]:
module = calc._module
module.eval()

all_e_mae_per_atom: list[float] = []
all_f_mae: list[float] = []

with torch.enable_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        with torch.no_grad():
            pass  # just move to device
        preds = module(batch)

        e_pred = preds["energy"].detach().cpu()       # (B,)
        e_true = batch.energy.cpu()                    # (B,)
        n_atoms = preds["num_atoms"].detach().cpu()   # (B,)

        e_mae = (torch.abs(e_pred - e_true) / n_atoms).mean().item()
        all_e_mae_per_atom.append(e_mae)

        if "forces" in preds and batch.forces is not None:
            f_pred = preds["forces"].detach().cpu()
            f_true = batch.forces.cpu()
            f_mae = torch.abs(f_pred - f_true).mean().item()
            all_f_mae.append(f_mae)

print("=== Test set results (OOD) ===")
print(f"Energy MAE/atom : {sum(all_e_mae_per_atom) / len(all_e_mae_per_atom):.4f} eV/atom")
if all_f_mae:
    print(f"Forces MAE      : {sum(all_f_mae) / len(all_f_mae):.4f} eV/Å")

---

## Summary

| Item | Value |
|---|---|
| Model | `nf_simurgh` — ARACE rounds + fragment virtual nodes (monolithic) |
| Checkpoint | best by `val_forces/forces_mae` (resolved from `checkpoint_state.json`) |
| Cutoff (atoms) | 8.0 Å |
| Cutoff (virtual nodes) | 15.0 Å |
| Elements | H, C, O |
| Fragment scheme | `custom`, cutting `[CX4]-[CX4]` and `[CX4]-[CX3]` |
| Head | `null` — monolithic, computes its own autograd forces |

### What differs from `simurgh_gmd_inference.ipynb`

1. **`NFSimurghCalculator`** (§1.1) — `GOALCalculator` plus a `fragment_index` on every
   graph, with the fragment settings read out of the checkpoint so inference partitions
   molecules exactly as training did. Without it, every inference call raises.
2. **§1.2** verifies the mechanism is live: fragment counts per structure, and whether
   the zero-initialised correction actually moved away from zero during training.
3. **§4 / §5** pin the decomposition (`freeze_fragments=True`) so relaxation and MD run
   on a continuous energy surface, and §5 checks whether that pinning mattered.
4. **§6.5** is a full report over **all 2,800 test structures** (1,400 OOD + 1,400 ID):
   per-molecule breakdown, error percentiles, worst frames, four diagnostic plots, and
   a per-structure CSV written next to the checkpoint. The old cell printed one line
   per structure plus a single mean — which hid a 20× spread across molecules.
5. Two bugs fixed along the way: the batch-inference cell resolved its data paths
   against the notebook directory (`FileNotFoundError`), and the split evaluation
   labelled a total energy error as "Energy MAE / atom" without dividing by the atom
   count.

`calc` is a standard ASE calculator — plug it into any ASE workflow. Beyond the usual
`energy` / `forces`, `calc.results` also carries `vnode_energies` and
`vnode_interaction_energies` (per fragment), and `calc.last_fragment_index` holds the
labels used by the most recent call.